In [1]:
import joblib
import pandas as pd
import numpy as np
import warnings

In [2]:
MODELS_PATH = '../src/models/models_to_use'

classifier = joblib.load(f'{MODELS_PATH}/xgbclassifier_threshold=0_34.joblib')
clusterizer = joblib.load(f'{MODELS_PATH}/clusteriser.joblib')
regressor_cluster_0 = joblib.load(f'{MODELS_PATH}/gboost_regressor_cluster=0.joblib')
regressor_cluster_1 = joblib.load(f'{MODELS_PATH}/xgb_regressor_cluster=1.joblib')
regressor_cluster_3 = joblib.load(f'{MODELS_PATH}/xgb_regressor_cluster=3.joblib')

df = pd.read_csv('../data/processed/startup_investment_dataset+rejected_status.csv')

In [3]:
def create_additional_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    eps = 1

    # --- Отношения денежных признаков ---
    df['requested_to_valuation'] = (df['requested_amount'] / (df['pre_money_valuation'] + eps))
    df['revenue_to_valuation'] = (df['annual_revenue'] / (df['pre_money_valuation'] + eps))
    df['revenue_per_employee'] = (df['annual_revenue'] / (df['team_size'] + eps))

    # --- Команда ---
    df['experience_per_member'] = (df['founders_experience_years'] / (df['team_size'] + eps))
    df['team_maturity'] = (df['team_size'] * df['founders_experience_years'])

    # --- Бизнес-флаги ---
    df['early_stage'] = (df['startup_stage'].isin(['Idea', 'Pre-Seed']).astype(int))
    df['is_us_market'] = ((df['region'] == 'US').astype(int))

    # --- Логарифмы денежных признаков ---

    numeric_features = df.select_dtypes(include=['int64', 'float64'])

    for col in numeric_features:
        if col in df.columns:
            df[f'{col}_log'] = np.log1p(df[col])

    # Очистка проблем после деления
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    return df

In [7]:
def predict_startup(startup: pd.Series | pd.DataFrame):
    startup = startup.copy()

    # превращаем в DataFrame
    if isinstance(startup, pd.Series):
        startup = startup.to_frame().T

    # FEATURE ENGINEERING
    startup = create_additional_features(startup)

    # STAGE 1: REJECTION CLASSIFICATION
    cls_features = startup.drop(
        columns=['investment_amount', 'is_rejected'],
        errors='ignore'
    )

    reject_prob = classifier.predict_proba(cls_features)[:, 1][0]
    is_rejected = int(reject_prob >= 0.34)

    print(f"Reject probability: {reject_prob:.2%}")

    if is_rejected:
        return {
            'status': 'REJECTED',
            'reject_probability': reject_prob
        }

    # STAGE 2: CLUSTERING
    cluster_features = [
        'requested_amount',
        'pre_money_valuation',
        'annual_revenue',
        'team_size'
    ]

    cluster = clusterizer.predict(startup[cluster_features])[0]
    print(f"Cluster: {cluster}")

    # STAGE 3: MODEL SELECTION
    models = {
        0: regressor_cluster_0,
        1: regressor_cluster_1,
        3: regressor_cluster_3,
        2: regressor_cluster_0  # редкий сегмент
    }

    model = models[cluster]

    # STAGE 4: REGRESSION
    X_reg = startup.drop(
        columns=[
            'investment_amount',
            'investment_amount_log',
            'cluster',
            'is_rejected'
        ],
        errors='ignore'
    )

    pred_log = model.predict(X_reg)[0]
    investment = np.expm1(pred_log)

    return {
        'status': 'APPROVED',
        'cluster': cluster,
        'predicted_investment': round(investment),
        'reject_probability': round(reject_prob, 4)
    }

In [20]:
for i in range(5):
    startup = df.sample(1)
    display(startup)
    
    result = predict_startup(startup)
    
    print(result)

    print('-' * 100)

,startup_stage,industry,region,requested_amount,pre_money_valuation,team_size,founders_experience_years,annual_revenue,market_size_estimate,investment_amount,is_rejected
1240,Series A,E-commerce,US,3.015911e+06,1.727501e+07,14,5,503817.711012,8.129841e+06,3.434347e+06,0


Reject probability: 17.53%
Cluster: 1
{'status': 'APPROVED', 'cluster': np.int32(1), 'predicted_investment': 2695983, 'reject_probability': np.float32(0.1753)}
----------------------------------------------------------------------------------------------------


,startup_stage,industry,region,requested_amount,pre_money_valuation,team_size,founders_experience_years,annual_revenue,market_size_estimate,investment_amount,is_rejected
791,Seed,AI/SaaS,Europe,2.772314e+06,1.996532e+07,9,4,321214.399894,1.901235e+06,0.0,1


Reject probability: 48.88%
{'status': 'REJECTED', 'reject_probability': np.float32(0.48884708)}
----------------------------------------------------------------------------------------------------


,startup_stage,industry,region,requested_amount,pre_money_valuation,team_size,founders_experience_years,annual_revenue,market_size_estimate,investment_amount,is_rejected
1626,Series A,AI/SaaS,US,2.102063e+06,1.342796e+07,11,11,365415.514071,1.403007e+07,1.340524e+06,0


Reject probability: 7.44%
Cluster: 1
{'status': 'APPROVED', 'cluster': np.int32(1), 'predicted_investment': 1715275, 'reject_probability': np.float32(0.0744)}
----------------------------------------------------------------------------------------------------


,startup_stage,industry,region,requested_amount,pre_money_valuation,team_size,founders_experience_years,annual_revenue,market_size_estimate,investment_amount,is_rejected
1201,Series B,E-commerce,Asia,1.543562e+07,8.707942e+07,8,17,189849.92631,1.420159e+07,1.736522e+07,0


Reject probability: 12.34%
Cluster: 0
{'status': 'APPROVED', 'cluster': np.int32(0), 'predicted_investment': 8141689, 'reject_probability': np.float32(0.1234)}
----------------------------------------------------------------------------------------------------


,startup_stage,industry,region,requested_amount,pre_money_valuation,team_size,founders_experience_years,annual_revenue,market_size_estimate,investment_amount,is_rejected
894,Seed,AI/SaaS,US,5.005502e+06,2.581968e+07,7,6,349897.773398,5.667565e+06,3.221860e+06,0


Reject probability: 28.09%
Cluster: 1
{'status': 'APPROVED', 'cluster': np.int32(1), 'predicted_investment': 3919604, 'reject_probability': np.float32(0.2809)}
----------------------------------------------------------------------------------------------------
